In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import quote

import numpy as np

os.environ["POLARS_OOC_MEMORY_BUDGET_MB"] = "100"  # disable Polars' own memory budget
os.environ["POLARS_ENGINE_AFFINITY"] = "streaming"
os.environ["POLARS_STREAMING_CHUNK_SIZE"] = "2000"  # small chunks to stress memory
os.environ["POLARS_VERBOSE"] = "0"  # enable verbose logging to stdout for debugging
os.environ["POLARS_MAX_THREADS"] = "1"  # enable verbose logging to stdout for debugging

import polars as pl
from data_warehousing_with_polars.incremental import _DeltaCdfSource, incremental
from deltalake import write_deltalake

_IMPL = Path(".").parent / "_memory_analysis_impl.py"
assert _IMPL.exists(), f"Expected {_IMPL} to exist"


def write_partitioned_measurements(
    src: Path, n_partitions: int, rows_per_partition: int, suffix: str = "00001"
) -> float:
    """Write partitioned parquet with (measurement, channel, value) columns.

    ``suffix`` names the file within each partition dir, so calling this again
    with a different suffix adds a genuinely new file for a later run to pick
    up (rather than overwriting the first batch — same path, same watermark
    entry, so it would never be seen as "new").
    """
    total_mbytes = 0.0
    for p in range(n_partitions):
        measurement = f"measurement_{p}"
        partition_dir = src / f"measurement={quote(measurement)}"
        partition_dir.mkdir(parents=True, exist_ok=True)
        df = pl.DataFrame([
            pl.repeat(measurement, rows_per_partition, eager=True).rename("measurement"),
            (pl.int_range(0, rows_per_partition, eager=True) % 5)
            .cast(pl.String)
            .str.pad_start(length=3, fill_char="0")
            .rename("channel"),
            pl.Series(np.random.rand(rows_per_partition)).rename("value"),
        ])
        df.write_parquet(partition_dir / f"{suffix}.parquet")
        total_mbytes += df.estimated_size(unit="mb")
    return total_mbytes


def cdf_seed(n_partitions: int, rows_per_partition: int) -> pl.DataFrame:
    return pl.concat([
        pl.DataFrame({
            "measurement": [f"measurement_{i}"] * rows_per_partition,
            "value": np.random.rand(rows_per_partition),
        })
        for i in range(n_partitions)
    ])


def _run_subprocess(name: str, tmp_path: Path, n_partitions: int, rows_per_partition: int) -> str:
    """Spawn a fresh interpreter to run ``run_<name>`` in ``_memory_analysis_impl.py``."""
    result = subprocess.run(
        [sys.executable, str(_IMPL), name, str(tmp_path), str(n_partitions), str(rows_per_partition)],
        capture_output=True,
        text=True,
        env={**os.environ},
    )
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Memory worker '{name}' failed:\n{output}")
    return output


def _measure(name: str, tmp_path: Path, n_partitions: int, rows_per_partition: int) -> dict:
    """Run the measured worker subprocess and parse its ``{"peak_rss_mb": ...}`` output.

    All dataset preparation happens in the notebook (this process) before this
    is called — see each ``prepare_*`` function below — so every worker
    subprocess only ever does the one operation being measured.
    """
    output = _run_subprocess(name, tmp_path, n_partitions, rows_per_partition)
    print(f"Output from memory worker '{name}':\n{output}")
    return json.loads(output.splitlines()[-1])


rows_per_partition = 100_000
num_partitions = 200


In [2]:
def prepare_src(tmp_path: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Writes the harness's first batch — the common case every worker starts from."""
    src = tmp_path / "src"
    src.mkdir(parents=True, exist_ok=True)
    return write_partitioned_measurements(src, n_partitions, rows_per_partition)


def prepare_by_partition_cdf(tmp_path: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Creates cdf_source, runs a throwaway pipeline once to establish the
    watermark/target, and appends the new CDF-visible commit the measured run
    picks up as its single-use batch. Returns that new batch's size (MB).
    """
    source = tmp_path / "cdf_source"
    write_deltalake(
        str(source),
        cdf_seed(n_partitions, rows_per_partition).to_arrow(),
        mode="overwrite",
        partition_by=["measurement"],
        configuration={"delta.enableChangeDataFeed": "true"},
    )

    @incremental(
        source=str(source),
        target=str(tmp_path / "target"),
        file_format="delta",
        merge_on=None,
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf: pl.LazyFrame) -> pl.LazyFrame:
        return lf

    pipeline.run()  # from_version=None, establishes the watermark

    new_batch = cdf_seed(n_partitions, rows_per_partition)
    write_deltalake(str(source), new_batch.to_arrow(), mode="append", partition_by=["measurement"])
    return new_batch.estimated_size(unit="mb")


def prepare_upsert(tmp_path: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Creates the table (a first run against src/), then writes a second file
    batch — the new data the measured run's ``_upsert_overwrite`` call processes.
    """
    prepare_src(tmp_path, n_partitions, rows_per_partition)

    @incremental(
        source=str(tmp_path / "src"),
        target=str(tmp_path / "target"),
        merge_on="channel",
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf: pl.LazyFrame) -> pl.LazyFrame:
        return lf

    pipeline.run()
    return write_partitioned_measurements(
        tmp_path / "src", n_partitions, rows_per_partition, suffix="00002"
    )


def prepare_fan_in_cdf_and_file(tmp_path: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Creates both sources, runs the pipeline once to establish both cursors,
    then writes new data to both — what the measured run processes.
    """
    prepare_src(tmp_path, n_partitions, rows_per_partition)
    cdf_source = tmp_path / "cdf_source"
    write_deltalake(
        str(cdf_source),
        cdf_seed(n_partitions, rows_per_partition).to_arrow(),
        mode="overwrite",
        partition_by=["measurement"],
        configuration={"delta.enableChangeDataFeed": "true"},
    )

    @incremental(
        source=[str(tmp_path / "src"), _DeltaCdfSource(str(cdf_source))],
        target=str(tmp_path / "target"),
        merge_on=None,
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf_file: pl.LazyFrame, lf_cdf: pl.LazyFrame) -> pl.LazyFrame:
        return pl.concat([lf_file, lf_cdf], how="diagonal_relaxed")

    pipeline.run()

    file_size = write_partitioned_measurements(
        tmp_path / "src", n_partitions, rows_per_partition, suffix="00002"
    )
    new_cdf_batch = cdf_seed(n_partitions, rows_per_partition)
    write_deltalake(
        str(cdf_source), new_cdf_batch.to_arrow(), mode="append", partition_by=["measurement"]
    )
    return file_size + new_cdf_batch.estimated_size(unit="mb")


def prepare_compact_every(
    tmp_path: Path, n_partitions: int, rows_per_partition: int, rounds: int = 3
) -> float:
    """Accumulates ``rounds`` small, uncompacted prior commits (simulating real
    file fragmentation from repeated small batches) before the measured run's
    ``compact_every=1`` triggers ``maintain()`` — otherwise there'd be nothing
    to compact on a freshly-created, already-tidy table.
    """
    prepare_src(tmp_path, n_partitions, rows_per_partition)

    @incremental(
        source=str(tmp_path / "src"),
        target=str(tmp_path / "target"),
        merge_on=None,
        partition_by="measurement",
        by_partition=True,
        by_partition_workers=1,
    )
    def pipeline(lf: pl.LazyFrame) -> pl.LazyFrame:
        return lf

    pipeline.run()  # first batch

    for r in range(rounds):
        write_partitioned_measurements(
            tmp_path / "src", n_partitions, rows_per_partition, suffix=f"prior{r:02d}"
        )
        pipeline.run()  # accumulates fragmentation; compact_every unset here on purpose

    # One more new batch for the measured subprocess's compact_every=1 run to process.
    return write_partitioned_measurements(
        tmp_path / "src", n_partitions, rows_per_partition, suffix="final"
    )


PREPARE = {
    "by_partition_cdf": prepare_by_partition_cdf,
    "upsert": prepare_upsert,
    "fan_in_cdf_and_file": prepare_fan_in_cdf_and_file,
    "compact_every": prepare_compact_every,
}


In [3]:
workers = [
    "pure_polars",
    "all",
    "by_partition",
    "scd2",
    "scd4",
    "by_partition_cdf",
    "upsert",
    "fan_in_cdf_and_file",
    "compact_every",
]

results = []

for i in range(1, num_partitions + 1, 20):
    for worker in workers:
        with TemporaryDirectory() as tmp_dir:
            tmp_path = Path(tmp_dir)
            prepare = PREPARE.get(worker, prepare_src)
            dataset_size_mb = prepare(tmp_path, i, rows_per_partition)

            result = _measure(worker, tmp_path, i, rows_per_partition)
            print(
                f"[{worker}] n_partitions={i}: "
                f"Peak RSS: {result['peak_rss_mb']:.1f} MB, "
                f"Dataset size: {dataset_size_mb:.1f} MB"
            )
            result.update({
                "dataset_size_mb": dataset_size_mb,
                "worker": worker,
                "n_partitions": i,
                "rows_per_partition": rows_per_partition,
            })
            results.append(result)

results_df = pl.DataFrame(results)


Output from memory worker 'pure_polars':
{"peak_rss_mb": 40.234375}
[pure_polars] n_partitions=1: Peak RSS: 40.2 MB, Dataset size: 2.3 MB
Output from memory worker 'all':
{"peak_rss_mb": 75.921875}
[all] n_partitions=1: Peak RSS: 75.9 MB, Dataset size: 2.3 MB


Output from memory worker 'by_partition':
{"peak_rss_mb": 80.109375}
[by_partition] n_partitions=1: Peak RSS: 80.1 MB, Dataset size: 2.3 MB
Output from memory worker 'scd2':
{"peak_rss_mb": 90.109375}
[scd2] n_partitions=1: Peak RSS: 90.1 MB, Dataset size: 2.3 MB


Output from memory worker 'scd4':
{"peak_rss_mb": 56.5}
[scd4] n_partitions=1: Peak RSS: 56.5 MB, Dataset size: 2.3 MB


Output from memory worker 'by_partition_cdf':
{"peak_rss_mb": 108.3125}
[by_partition_cdf] n_partitions=1: Peak RSS: 108.3 MB, Dataset size: 2.0 MB


Output from memory worker 'upsert':
{"peak_rss_mb": 122.390625}
[upsert] n_partitions=1: Peak RSS: 122.4 MB, Dataset size: 2.3 MB


...

Output from memory worker 'scd4':
{"peak_rss_mb": 108.6875}
[scd4] n_partitions=181: Peak RSS: 108.7 MB, Dataset size: 438.3 MB


Output from memory worker 'by_partition_cdf':
{"peak_rss_mb": 412.078125}
[by_partition_cdf] n_partitions=181: Peak RSS: 412.1 MB, Dataset size: 386.5 MB


Output from memory worker 'upsert':
{"peak_rss_mb": 362.96875}
[upsert] n_partitions=181: Peak RSS: 363.0 MB, Dataset size: 438.3 MB


Output from memory worker 'fan_in_cdf_and_file':
{"peak_rss_mb": 467.15625}
[fan_in_cdf_and_file] n_partitions=181: Peak RSS: 467.2 MB, Dataset size: 824.8 MB


Output from memory worker 'compact_every':
{"peak_rss_mb": 846.984375}
[compact_every] n_partitions=181: Peak RSS: 847.0 MB, Dataset size: 438.3 MB


In [4]:
import altair as alt

# Each incremental code path gets its own identifiable color (Tableau-10-style,
# avoiding blue/black — those are reserved below); polars stays a fixed
# reference blue; dataset size is its own thin dashed black reference line.
_worker_domain = [
    "pure_polars",
    "all",
    "by_partition",
    "scd2",
    "scd4",
    "by_partition_cdf",
    "upsert",
    "fan_in_cdf_and_file",
    "compact_every",
    "dataset_size",
]
_color_range = [
    "#0075ff",  # pure_polars — polars blue
    "#F28E2B",  # all
    "#59A14F",  # by_partition
    "#E15759",  # scd2
    "#EDC948",  # scd4
    "#B07AA1",  # by_partition_cdf
    "#FF9DA7",  # upsert
    "#9C755F",  # fan_in_cdf_and_file
    "#BAB0AC",  # compact_every
    "#000000",  # dataset_size
]
# Every code path drawn the same weight — no bold treatment; only dataset
# size is deliberately thinner, since it's a reference line, not a result.
_width_range = [2, 2, 2, 2, 2, 2, 2, 2, 2, 1]
_dash_range = [[1, 0]] * 9 + [[6, 4]]  # solid for every worker, dashed for dataset size

long_rows = []
for r in results_df.iter_rows(named=True):
    long_rows.append({
        "n_partitions": r["n_partitions"],
        "worker": r["worker"],
        "series": r["worker"],
        "value": r["peak_rss_mb"],
    })
    long_rows.append({
        "n_partitions": r["n_partitions"],
        "worker": r["worker"],
        "series": "dataset_size",
        "value": r["dataset_size_mb"],
    })

long_df = pl.DataFrame(long_rows)

alt.Chart(long_df).mark_line().encode(
    x=alt.X("n_partitions:Q", title="n_partitions"),
    y=alt.Y("value:Q", title="peak RSS / dataset size (MB)"),
    color=alt.Color(
        "series:N",
        scale=alt.Scale(domain=_worker_domain, range=_color_range),
        legend=alt.Legend(title=None),
    ),
    strokeWidth=alt.StrokeWidth(
        "series:N", scale=alt.Scale(domain=_worker_domain, range=_width_range), legend=None
    ),
    strokeDash=alt.StrokeDash(
        "series:N", scale=alt.Scale(domain=_worker_domain, range=_dash_range), legend=None
    ),
    detail="worker:N",
    tooltip=["worker", "n_partitions", "value", "series"],
).properties(width=700, height=420)


alt.Chart(...)

In [5]:
results_df.sort("n_partitions", "worker")


peak_rss_mb,dataset_size_mb,worker,n_partitions,rows_per_partition
f64,f64,str,i64,i64
75.921875,2.288818,"""all""",1,100000
80.109375,2.288818,"""by_partition""",1,100000
108.3125,2.002716,"""by_partition_cdf""",1,100000
185.53125,2.288818,"""compact_every""",1,100000
150.640625,4.291534,"""fan_in_cdf_and_file""",1,100000
…,…,…,…,…
467.15625,824.832916,"""fan_in_cdf_and_file""",181,100000
679.84375,438.308716,"""pure_polars""",181,100000
363.890625,438.308716,"""scd2""",181,100000
